# Lesson 5B: N-Step Methods & Eligibility Traces - Practical

## Introduction

In Lesson 5A, we learned TD(λ) and eligibility traces—the mathematical framework for n-step returns.

Now we implement **Sarsa(λ)** on **RandomWalk**—the canonical environment for seeing how λ trades off bias and variance.

### RandomWalk Problem

- 1D chain: states 0-6 (left to right)
- Start at state 3
- Actions: left (-1) or right (+1)
- Reward: 0 except terminal (−1 at left, +1 at right)
- True values: V(0)=0, V(1)=1/6, V(2)=2/6, ..., V(6)=1

Perfect for analyzing prediction error because ground truth is known analytically.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

np.random.seed(42)
print('N-Step Eligibility Traces Practical')

## RandomWalk Environment (Gymnasium API)

In [ ]:
import gymnasium as gym
from gymnasium import spaces

class RandomWalk(gym.Env):
    """1D random walk using Gymnasium API.
    
    States: 0-6 (left to right), start at 3.
    Rewards: -1 at left boundary, +1 at right boundary, 0 otherwise.
    True values: V(s) = s/6 (linearly interpolated).
    """
    
    def __init__(self, n_states=7):
        self.n_states = n_states
        self.start_state = n_states // 2
        self.state = self.start_state
        self.true_values = np.linspace(0, 1, n_states)
        
        # Gymnasium spaces
        self.action_space = spaces.Discrete(2)  # 0=left, 1=right
        self.observation_space = spaces.Discrete(n_states)
    
    def reset(self, seed=None):
        super().reset(seed=seed)
        self.state = self.start_state
        return self.state, {}
    
    def step(self, action):
        assert self.action_space.contains(action), f"Invalid action {action}"
        
        if action == 0:
            self.state -= 1
        else:
            self.state += 1
        
        done = self.state < 0 or self.state >= self.n_states
        reward = 0.0
        
        if self.state < 0:
            reward = -1.0
        elif self.state >= self.n_states:
            reward = 1.0
        
        # For step computation, return None state if terminal (to maintain logic)
        return_state = self.state if not done else self.start_state
        
        return return_state, reward, done, False, {}

env = RandomWalk()
state, _ = env.reset()
print(f'True values: {env.true_values}')
print(f'Gymnasium spaces - Action: {env.action_space}, Observation: {env.observation_space}')

## Sarsa(λ) Implementation

In [ ]:
def sarsa_lambda(env, lam=0.5, alpha=0.1, gamma=0.99, episodes=100, trace_type='accumulating'):
    """
    Sarsa(λ) on RandomWalk.
    
    Args:
        trace_type: 'accumulating' (e += 1) or 'replacing' (e = 1)
    """
    V = np.zeros(env.observation_space.n)
    e = np.zeros(env.observation_space.n)
    errors = []
    
    for ep in range(episodes):
        s, _ = env.reset()
        e.fill(0)
        episode_error = 0.0
        step_count = 0
        
        while step_count < 1000:
            a = env.action_space.sample()  # Random action
            s_next, r, done, _, _ = env.step(a)
            step_count += 1
            
            v_next = V[s_next] if not done else 0.0
            delta = r + gamma * v_next - V[s]
            
            # Update trace
            if trace_type == 'accumulating':
                e[s] += 1.0
            else:  # replacing
                e[s] = 1.0
            
            # Update all states
            for i in range(env.observation_space.n):
                V[i] += alpha * delta * e[i]
                e[i] *= gamma * lam
            
            episode_error += np.abs(delta)
            
            if done:
                break
            s = s_next
        
        errors.append(episode_error)
    
    return V, np.array(errors)

# Test with different lambda values and trace types
print('Testing Sarsa(λ) - Accumulating traces:')
lambdas = [0.0, 0.3, 0.7, 1.0]
results_accum = {}

for lam in lambdas:
    V_learned, errors = sarsa_lambda(env, lam=lam, episodes=200, trace_type='accumulating')
    mse = np.mean((V_learned - env.true_values) ** 2)
    results_accum[lam] = (V_learned, errors, mse)
    print(f'λ={lam}: MSE={mse:.4f}')

print('\nTesting Sarsa(λ) - Replacing traces:')
results_replace = {}

for lam in lambdas:
    V_learned, errors = sarsa_lambda(env, lam=lam, episodes=200, trace_type='replacing')
    mse = np.mean((V_learned - env.true_values) ** 2)
    results_replace[lam] = (V_learned, errors, mse)
    print(f'λ={lam}: MSE={mse:.4f}')

## Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Accumulating trace: Learning curves
ax = axes[0, 0]
for lam, (_, errors, _) in results_accum.items():
    window = 20
    smoothed = np.convolve(errors, np.ones(window)/window, mode='valid')
    ax.plot(smoothed, label=f'λ={lam}', linewidth=2)
ax.set_xlabel('Episode')
ax.set_ylabel('TD Error (Smoothed)')
ax.set_title('Accumulating Traces - Learning Curves')
ax.legend()
ax.grid(True, alpha=0.3)

# Replacing trace: Learning curves
ax = axes[0, 1]
for lam, (_, errors, _) in results_replace.items():
    window = 20
    smoothed = np.convolve(errors, np.ones(window)/window, mode='valid')
    ax.plot(smoothed, label=f'λ={lam}', linewidth=2)
ax.set_xlabel('Episode')
ax.set_ylabel('TD Error (Smoothed)')
ax.set_title('Replacing Traces - Learning Curves')
ax.legend()
ax.grid(True, alpha=0.3)

# Accumulating: Value function comparison
ax = axes[1, 0]
ax.plot(env.true_values, 'ko-', label='True V(s)', linewidth=3, markersize=8)
for lam, (V_learned, _, _) in results_accum.items():
    ax.plot(V_learned, 'o-', label=f'λ={lam}', alpha=0.7, linewidth=2, markersize=6)
ax.set_xlabel('State')
ax.set_ylabel('Value')
ax.set_title('Accumulating Traces - Learned vs True Values')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Replacing: Value function comparison
ax = axes[1, 1]
ax.plot(env.true_values, 'ko-', label='True V(s)', linewidth=3, markersize=8)
for lam, (V_learned, _, _) in results_replace.items():
    ax.plot(V_learned, 'o-', label=f'λ={lam}', alpha=0.7, linewidth=2, markersize=6)
ax.set_xlabel('State')
ax.set_ylabel('Value')
ax.set_title('Replacing Traces - Learned vs True Values')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Trace-decay comparison complete.')

## Key Findings

### RandomWalk (Tabular, Gymnasium API)

1. **λ=0 (TD(0))**: High bias, fast convergence
2. **λ=1 (MC)**: High variance, slow convergence
3. **λ∈(0,1)**: Sweet spot—balanced bias/variance
4. **Trace types**:
   - **Accumulating** (e += 1): Emphasizes recency
   - **Replacing** (e = 1): Single visit per state per episode

### MountainCar (Continuous, Function Approximation)

- Sarsa(λ) with tile coding scales to continuous states
- Eligibility traces efficiently credit multiple tiles (basis functions)
- Shows the bridge between tabular RL and deep RL

### Framework Comparison

- **From-scratch Sarsa(λ)**: Transparent, educational, direct control
- **Stable-Baselines3 DQN**: Production-ready, robust, handles large state spaces

Both achieve the same goal—interpolating TD and MC—via different trade-offs.

### Why Eligibility Traces Matter

1. **Efficient credit assignment**: All visited states updated immediately
2. **Flexible n-step returns**: Single parameter λ controls the spectrum
3. **Bridge to modern RL**: Eligibility traces concepts appear in:
   - Experience replay prioritization
   - Importance sampling weights
   - GAE (Generalized Advantage Estimation) in PPO/A3C

### Next: Lesson 6

**Function Approximation**—neural networks instead of tile coding, scaling to high-dimensional problems.

In [ ]:
from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import BaseCallback

# Train Stable-Baselines3 DQN on MountainCar
print("Training Stable-Baselines3 DQN on MountainCar...")
model = DQN(
    'MlpPolicy',
    env_mc,
    learning_rate=0.001,
    buffer_size=10000,
    exploration_fraction=0.1,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.05,
    verbose=0,
)

# Store episode returns
class EpisodeRewardLogger(BaseCallback):
    def __init__(self):
        super().__init__()
        self.episode_rewards = []
        self.episode_count = 0
    
    def _on_step(self) -> bool:
        if self.model.num_timesteps % 200 == 0:
            self.episode_rewards.append(self.model.ep_info_buffer[-1]['r'] if len(self.model.ep_info_buffer) > 0 else 0)
        return True

callback = EpisodeRewardLogger()
model.learn(total_timesteps=20000, callback=callback, progress_bar=False)

print(f"DQN training complete. Final return: {callback.episode_rewards[-1]:.1f}")

# Compare returns
fig, ax = plt.subplots(figsize=(10, 5))

# Rescale our Sarsa(λ) returns for fair comparison (our episodes may be different length)
window = 10
our_smoothed = np.convolve(returns_mc, np.ones(window)/window, mode='valid')

# Plot both
ax.plot(our_smoothed, label='Our Sarsa(λ)', linewidth=2, alpha=0.7)
if len(callback.episode_rewards) > 0:
    ax.plot(callback.episode_rewards, label='Stable-Baselines3 DQN', linewidth=2, alpha=0.7)

ax.set_xlabel('Episode')
ax.set_ylabel('Return')
ax.set_title('From-Scratch Sarsa(λ) vs Stable-Baselines3 DQN on MountainCar')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nKey difference:")
print("- Sarsa(λ): On-policy, uses eligibility traces for credit assignment")
print("- DQN: Off-policy, uses replay buffer + target network for stability")
print("Both interpolate between TD and MC via different mechanisms!")

## Stable-Baselines3 Reference: QR-DQN (Double Q-Learning with Eligibility Traces)

Compare our from-scratch Sarsa(λ) against Stable-Baselines3's production agent (using double Q-learning with eligibility trace concepts baked in).

In [ ]:
# Load MountainCar from Gymnasium
env_mc = gym.make('MountainCar-v0')
print(f"MountainCar observation space: {env_mc.observation_space}")
print(f"MountainCar action space: {env_mc.action_space}")

# Simple tile coding for function approximation
def get_tiles(state, num_tilings=8, tile_size=10):
    """Tile code a continuous state into binary features."""
    position, velocity = state
    tiles = []
    for i in range(num_tilings):
        offset = i / num_tilings
        pos_tile = int((position + offset - (-1.2)) / (0.6 + 1) * tile_size)
        vel_tile = int((velocity + offset - (-0.07)) / (0.14) * tile_size)
        tiles.append((min(max(pos_tile, 0), tile_size-1), min(max(vel_tile, 0), tile_size-1)))
    return tiles

# Sarsa(λ) with tile coding
def sarsa_lambda_tiles(env_mc, lam=0.5, alpha=0.1, episodes=100):
    """Sarsa(λ) on MountainCar with tile coding."""
    num_tilings = 8
    tile_size = 10
    num_features = num_tilings * tile_size * tile_size
    
    w = np.zeros((env_mc.action_space.n, num_features))  # Weights
    e = np.zeros((env_mc.action_space.n, num_features))  # Traces
    returns = []
    
    for ep in range(episodes):
        state, _ = env_mc.reset()
        e.fill(0)
        
        # ε-greedy: pick action
        if np.random.rand() < 0.1:
            a = env_mc.action_space.sample()
        else:
            a = np.argmax([np.sum(w[a_]) for a_ in range(env_mc.action_space.n)])
        
        episode_return = 0
        done = False
        
        for _ in range(1000):
            state_next, r, done, _, _ = env_mc.step(a)
            episode_return += r
            
            # Get feature representation
            tiles = get_tiles(state)
            features = np.zeros(num_features)
            for tile in tiles:
                features[tile[0] * tile_size + tile[1]] = 1.0
            
            # Pick next action
            if np.random.rand() < 0.1:
                a_next = env_mc.action_space.sample()
            else:
                a_next = np.argmax([np.sum(w[a_]) for a_ in range(env_mc.action_space.n)])
            
            # TD error and update
            tiles_next = get_tiles(state_next)
            features_next = np.zeros(num_features)
            for tile in tiles_next:
                features_next[tile[0] * tile_size + tile[1]] = 1.0
            
            v = np.sum(w[a] * features)
            v_next = np.sum(w[a_next] * features_next) if not done else 0
            delta = r + 0.99 * v_next - v
            
            # Update traces and weights
            e[a] += features
            for a_ in range(env_mc.action_space.n):
                w[a_] += alpha * delta * e[a_]
                e[a_] *= 0.99 * lam
            
            if done:
                break
            
            state = state_next
            a = a_next
        
        returns.append(episode_return)
    
    return w, np.array(returns)

# Train on MountainCar
print("Training Sarsa(λ) on MountainCar (this may take a moment)...")
w_mc, returns_mc = sarsa_lambda_tiles(env_mc, lam=0.9, episodes=100)
print(f"Average return (last 10 episodes): {np.mean(returns_mc[-10:]):.1f}")

# Plot
plt.figure(figsize=(10, 5))
window = 10
smoothed = np.convolve(returns_mc, np.ones(window)/window, mode='valid')
plt.plot(smoothed, linewidth=2)
plt.xlabel('Episode')
plt.ylabel('Return')
plt.title('Sarsa(λ) on MountainCar (λ=0.9, Tile Coding)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## MountainCar: Continuous States

Now scale to a continuous environment. MountainCar requires function approximation, foreshadowing Lesson 6.

We'll use tile coding—a simple linear function approximator.